# ⚽ Player Data Analytics — 2026/27 Season
## Full 7-Step Analytical Workflow

**Dataset:** FBref · Europe's Top 5 Leagues · Season 2026/27  
**Source:** [Kaggle — Football Players Stats 2026-2027](https://www.kaggle.com/datasets/hubertsidorowicz/football-players-stats-2026-2027)  

---

### Workflow Steps
| Step | Description |
|------|-------------|
| 1 | **Environment Setup & Data Loading** |
| 2 | **Exploratory Data Analysis (EDA)** |
| 3 | **Feature Engineering** |
| 4 | **Attacking Analysis** |
| 5 | **Defensive & Disciplinary Analysis** |
| 6 | **Goalkeeping Analysis** |
| 7 | **Predictive Modeling — High Performer Classification** |

## Step 1 · Environment Setup & Data Loading

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..', 'dashboard'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from utils import load_raw_data, clean_data, kpi_summary, league_summary, LEAGUE_COLORS

plt.rcParams.update({
    'figure.facecolor': '#0e1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'text.color':       '#c9d1d9',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'grid.color':       '#21262d',
    'figure.titlesize': 14,
})

DATA_PATH = os.path.join('..', 'data', 'Player_data.csv')
print('Data path:', os.path.abspath(DATA_PATH))

In [ ]:
df_raw = load_raw_data(DATA_PATH)
print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
df = clean_data(df_raw)
print(f'Cleaned shape: {df.shape}')
print('Leagues:', df['League'].unique().tolist())
print('Positions:', df['PrimaryPos'].unique().tolist())

## Step 2 · Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
print(f'Rows:    {len(df):,}')
print(f'Columns: {df.shape[1]}')
print(f'Players: {df["Player"].nunique():,}')
print(f'Nations: {df["Nation"].nunique()}')
print(f'Clubs:   {df["Squad"].nunique()}')
print()
print('=== KPI Summary ===')
for k, v in kpi_summary(df).items():
    print(f'  {k:<20} {v}')

In [ ]:
miss = df.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)
print('Columns with missing values:')
print(miss)

In [ ]:
key_cols = ['Age','Min','90s','Gls','Ast','G+A','Sh','SoT','TklW','Int','CrdY','CrdR']
df[key_cols].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('Distribution of Key Numeric Features', fontsize=14, color='#f0f6fc', y=1.02)
plot_cols = ['Age','Min','Gls','Ast','Sh','SoT','TklW','Int']
for ax, col in zip(axes.flat, plot_cols):
    data = df[col].dropna()
    ax.hist(data, bins=30, color='#58a6ff', alpha=0.8, edgecolor='#0e1117')
    ax.set_title(col, color='#c9d1d9', fontsize=11)
    ax.set_xlabel('')
plt.tight_layout()
plt.savefig(os.path.join('..','outputs','charts','eda_distributions.png'),
            dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()

In [ ]:
corr_cols = ['Age','Min','Gls','Ast','G+A','Sh','SoT','SoT%','TklW','Int','CrdY','Fld','Fls']
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, linecolor='#30363d',
            ax=ax, cbar_kws={'shrink': .8})
ax.set_title('Feature Correlation Matrix', color='#f0f6fc', pad=12)
plt.tight_layout()
plt.savefig(os.path.join('..','outputs','charts','correlation_matrix.png'),
            dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()

## Step 3 · Feature Engineering

In [ ]:
# Engineered features are created in utils.clean_data()
eng_cols = ['Gls_90','Ast_90','GA_90_off','InvolvementScore',
            'DefWork_90','DisciplineIdx','ShotAcc','AgeGroup']
print('Engineered features:')
df[eng_cols].describe().round(3)

In [ ]:
ag = df.groupby('AgeGroup', observed=True).agg(
    Players=('Player','count'),
    Goals=('Gls','sum'),
    Assists=('Ast','sum'),
    AvgGls90=('Gls_90','mean'),
).reset_index().round(3)
print('Stats by Age Group:')
ag

In [ ]:
df_out = df[df['PrimaryPos'] != 'GK'].copy()
df_out = df_out[df_out['90s'].fillna(0) >= 3]
threshold = df_out['GA_90_off'].quantile(0.75)
df_out['HighPerformer'] = (df_out['GA_90_off'] >= threshold).astype(int)
print(f'HP threshold (top 25% G+A/90): {threshold:.3f}')
print(df_out['HighPerformer'].value_counts())

## Step 4 · Attacking Analysis

In [ ]:
top_scorers = df[df['PrimaryPos']!='GK'].nlargest(10,'Gls')[['Player','Squad','League','Gls','Ast','G+A']]
print('=== Top 10 Scorers ===')
print(top_scorers.to_string(index=False))

In [ ]:
df_eff = df[(df['Sh'] >= 10) & df['G/Sh'].notna()]
fig = px.scatter(df_eff, x='Sh', y='Gls', color='League',
                 hover_name='Player', size='Min', size_max=20,
                 title='Goals vs Shots (min. 10 shots)',
                 template='plotly_dark')
fig.update_layout(height=450)
fig.show()

In [ ]:
lg = league_summary(df)
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(lg))
w = 0.35
ax.bar([i - w/2 for i in x], lg['Goals'],   width=w, label='Goals',   color='#58a6ff')
ax.bar([i + w/2 for i in x], lg['Assists'],  width=w, label='Assists', color='#3fb950')
ax.set_xticks(list(x))
ax.set_xticklabels(lg['League'].tolist(), rotation=15, ha='right')
ax.set_ylabel('Count')
ax.set_title('Goals & Assists by League', color='#f0f6fc', pad=10)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join('..','outputs','charts','goals_assists_league.png'),
            dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()

In [ ]:
# Shot-on-Target % by position
sot_pos = (df[(df['Sh'] >= 5) & (df['PrimaryPos']!='GK')]
           .groupby('PrimaryPos', observed=True)['SoT%'].mean()
           .sort_values(ascending=False))
print('Avg SoT% by Position:')
print(sot_pos.round(2))

## Step 5 · Defensive & Disciplinary Analysis

In [ ]:
df_def = df[df['PrimaryPos'] != 'GK'].copy()
top_def = df_def.nlargest(10,'TklW')[['Player','Squad','League','PrimaryPos','TklW','Int']]
print('=== Top 10 Tacklers ===')
print(top_def.to_string(index=False))

In [ ]:
df_dm = df_def[(df_def['PrimaryPos'].isin(['DF','MF'])) & (df_def['Min'].fillna(0) >= 270)]
fig = px.scatter(df_dm, x='TklW', y='Int', color='PrimaryPos',
                 hover_name='Player', title='Tackles Won vs Interceptions',
                 template='plotly_dark',
                 color_discrete_map={'DF':'#58a6ff','MF':'#3fb950'})
fig.update_layout(height=420)
fig.show()

In [ ]:
disc = (df.groupby('League', observed=True)
          .agg(YellowCards=('CrdY','sum'), RedCards=('CrdR','sum'))
          .reset_index())
print('Disciplinary stats by League:')
print(disc.to_string(index=False))

In [ ]:
foul_lg = (df_def.groupby('League', observed=True)
           .agg(FoulsDrawn=('Fld','mean'), FoulsCommitted=('Fls','mean'))
           .reset_index().round(2))
print('Avg Fouls per player by League:')
print(foul_lg.to_string(index=False))

## Step 6 · Goalkeeping Analysis

In [ ]:
from utils import gk_summary
gk = gk_summary(df, min_mins=270)
print(f'Qualified GKs: {len(gk)}')
print('Top 10 by Save%:')
print(gk.head(10)[['Player','Squad','League','Save%','GA90','CS','CS%']].to_string(index=False))

In [ ]:
fig = px.scatter(gk.dropna(subset=['Save%','GA90']),
                 x='GA90', y='Save%', color='League',
                 hover_name='Player', title='GK: Save% vs Goals Allowed/90',
                 template='plotly_dark')
fig.update_layout(height=420)
fig.show()

In [ ]:
cs_lg = (gk.dropna(subset=['CS%'])
           .groupby('League', observed=True)['CS%'].mean()
           .sort_values(ascending=False).reset_index().round(1))
print('Avg Clean Sheet % by League:')
print(cs_lg.to_string(index=False))

## Step 7 · Predictive Modeling — High Performer Classification

In [ ]:
import sys
sys.path.insert(0, os.path.join('..', 'dashboard'))
from modeling import train_model, save_model, plot_feature_importance, plot_confusion_matrix, plot_roc_curve, plot_cv_comparison
import modeling
modeling.CHART_DIR = os.path.join('..', 'outputs', 'charts')

print('Training models (this may take ~30 seconds)...')
results = train_model(df)

In [ ]:
model_out = os.path.join('..', 'outputs', 'Player_data_model.joblib')
save_model(results, model_out)
print(f'Model saved to {model_out}')

In [ ]:
fi_path  = plot_feature_importance(results)
cm_path  = plot_confusion_matrix(results)
roc_path = plot_roc_curve(results)
cv_path  = plot_cv_comparison(results)
print('Charts saved:')
for p in [fi_path, cm_path, roc_path, cv_path]:
    if p: print(' ', p)

In [ ]:
from IPython.display import Image, display
for p in [fi_path, cm_path, roc_path, cv_path]:
    if p and os.path.exists(p):
        display(Image(filename=p))

In [ ]:
import pandas as pd
rep = results['report']
rep_df = pd.DataFrame(rep).T.drop(columns=['support'], errors='ignore').round(3)
print(f"Best Model  : {results['model_name']}")
print(f"ROC-AUC     : {results['roc_auc']:.4f}")
print()
print(rep_df)

In [ ]:
# Save cleaned dataset to outputs/
from utils import save_cleaned
out_csv = os.path.join('..', 'outputs', 'cleaned_Player_data.csv')
save_cleaned(df, out_csv)

## Conclusion

### Key Findings
1. **La Liga** leads in attacking output (goals & assists) among the five leagues.
2. **Forwards** have the highest shot-on-target %, but midfielders contribute significantly to assists.
3. **Defensive Work**: Central defenders and defensive midfielders dominate tackles + interceptions.
4. **Age**: Peak performance cluster is in the **26–29** age group for both attacking and defensive metrics.
5. **ML Model**: The Random Forest achieves strong ROC-AUC performance, with `Sh/90`, `G/Sh`, `Gls`, and `Min` as top predictors of high-performer status.

### Next Steps
- Incorporate xG / xA data when available
- Expand to full dataset (`players_data-2026_2027.csv`) for deeper passing/possession analysis
- Time-series analysis as the season progresses
- Clustering players by playing style using unsupervised learning